# W04 — Verify Baseline (self-contained)

Chạy MỘT LẦN (**Runtime > Run all**) để xác nhận baseline **Qwen2.5-VL-3B zero-shot** trên ViTextVQA = **EM 38.3% / ANLS 55.9%**.

**TRƯỚC KHI CHẠY:** Runtime > Change runtime type > **T4 GPU**.

Notebook TỰ CHỨA: không cần clone repo; metric nhúng sẵn (đúng bản cho 38.3/55.9); data lấy từ Drive (đã tải ở W4, không tải lại). Thời gian: ~10 phút.

In [ ]:
# --- 1. Kiểm GPU (dừng sớm với thông báo rõ nếu chưa bật) ---
import torch
assert torch.cuda.is_available(), (
    '❌ CHUA BAT GPU. Vao: Runtime > Change runtime type > T4 GPU, '
    'roi Runtime > Run all lai.'
)
print('✅ GPU:', torch.cuda.get_device_name(0))

In [ ]:
# --- 2. Cai dependencies (pillow<12 de khong xung dot torchvision) ---
!pip install -q -U "transformers>=4.49.0" "qwen-vl-utils" accelerate bitsandbytes "pillow<12"
print('✅ Cai xong. Neu sau do import loi -> Runtime > Restart runtime roi Run all lai.')

In [ ]:
# --- 3. Mount Drive + duong dan + seed + log version ---
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json, random, shutil, subprocess, importlib.metadata as meta
import torch

PROJECT_DIR = '/content/drive/MyDrive/Colab Notebooks/ViVQA-VLM'
DATA_DIR = os.path.join(PROJECT_DIR, 'data', 'vitextvqa')
EXP_DIR  = os.path.join(PROJECT_DIR, 'experiments', 'W04_verify_qwen25vl_seed42')
os.makedirs(EXP_DIR, exist_ok=True)
MODEL_ID = 'Qwen/Qwen2.5-VL-3B-Instruct'
SEED = 42
random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

# GPU log — guard de KHONG sap neu thieu nvidia-smi
if shutil.which('nvidia-smi'):
    print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,memory.used','--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip())
def _v(p):
    try: return meta.version(p)
    except Exception: return 'NA'
print('transformers', _v('transformers'), '| torch', torch.__version__, '| pillow', _v('pillow'))
print('PROJECT_DIR ton tai?', os.path.isdir(PROJECT_DIR))

In [ ]:
# --- 4. Dam bao co test.json + anh (dung ban tren Drive tu W4; tai neu thieu) ---
from huggingface_hub import hf_hub_download
import zipfile

REPO = 'nhonhoccode/ViTextVQA'
os.makedirs(DATA_DIR, exist_ok=True)
test_json = os.path.join(DATA_DIR, 'test.json')
if not os.path.exists(test_json):
    print('Tai test.json...')
    shutil.copy(hf_hub_download(REPO, 'test.json', repo_type='dataset'), test_json)

IMG_DIR = os.path.join(DATA_DIR, 'images')
def find_image_root(root):
    if os.path.isdir(root):
        for r, _, fs in os.walk(root):
            if any(f.lower().endswith('.jpg') for f in fs):
                return r
    return None
IMG_ROOT = find_image_root(IMG_DIR)
if IMG_ROOT is None:
    print('Tai images.zip (5.59GB, chi lan dau)...')
    z = hf_hub_download(REPO, 'images.zip', repo_type='dataset', cache_dir=os.path.join(DATA_DIR,'_hf_cache'))
    with zipfile.ZipFile(z) as zf: zf.extractall(IMG_DIR)
    IMG_ROOT = find_image_root(IMG_DIR)
print('Anh o:', IMG_ROOT)

In [ ]:
# --- 5. Join COCO-style + shuffle seed 42 + lay 300 (khop dung run W4) ---
with open(test_json, encoding='utf-8') as f:
    d = json.load(f)
id2file = {im['id']: im['filename'] for im in d['images']}
flat = []
for a in d['annotations']:
    fn = id2file.get(a['image_id'])
    if fn is None:
        continue
    flat.append({'question_id': a['id'], 'question': a['question'],
                 'answers': a['answers'], 'image': fn})
random.Random(42).shuffle(flat)
subset = flat[:300]
print('Tong annotation:', len(flat), '| dung:', len(subset))

In [ ]:
# --- 6. Load model + processor (T4 -> float16) ---
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

print('Load model (lan dau ~2-3 phut)...')
processor = AutoProcessor.from_pretrained(MODEL_ID, max_pixels=1024*28*28)
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, dtype=torch.float16, device_map='auto', attn_implementation='sdpa')
model.eval()
print('dtype:', next(model.parameters()).dtype)

In [ ]:
# --- 7. Metric NHUNG SAN (dung ban cho 38.3/55.9) ---
import re, unicodedata
def normalize(s):
    s = unicodedata.normalize('NFC', str(s).lower().strip())
    s = re.sub(r'[^\w\s]', ' ', s, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', s).strip()
def lev(a, b):
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
    return prev[-1]
def em(pred, golds):
    p = normalize(pred)
    return float(any(p == normalize(g) for g in golds))
def anls(pred, golds, tau=0.5):
    p = normalize(pred); best = 0.0
    for g in golds:
        g = normalize(g); nl = lev(p, g)/max(len(p), len(g), 1); s = 1-nl
        best = max(best, s if s >= tau else 0.0)
    return best
print('✅ metric san sang')

In [ ]:
# --- 8. Zero-shot 300 mau + cham diem ---
import time
from PIL import Image

PROMPT = ("Trả lời câu hỏi bằng tiếng Việt, NGẮN GỌN, chỉ nêu đáp án "
          "dựa trên chữ và nội dung trong ảnh. Câu hỏi: {q}")
results = []
t0 = time.time()
for i, s in enumerate(subset):
    p = os.path.join(IMG_ROOT, s['image'])
    if not os.path.exists(p):
        continue
    img = Image.open(p).convert('RGB')
    msg = [{'role':'user','content':[{'type':'image','image':img},
            {'type':'text','text':PROMPT.format(q=s['question'])}]}]
    text = processor.apply_chat_template(msg, tokenize=False, add_generation_prompt=True)
    image_inputs, _ = process_vision_info(msg)
    inputs = processor(text=[text], images=image_inputs, padding=True,
                       return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
    pred = processor.batch_decode(out[:, inputs.input_ids.shape[1]:],
                                  skip_special_tokens=True)[0].strip()
    results.append({'question_id': s['question_id'], 'question': s['question'],
                    'gold': s['answers'], 'pred': pred, 'image': s['image']})
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{len(subset)}  ({(time.time()-t0)/(i+1):.2f}s/mau)')

json.dump(results, open(os.path.join(EXP_DIR,'verify_n300.json'),'w',encoding='utf-8'), ensure_ascii=False)

EM = sum(em(r['pred'], r['gold']) for r in results)/len(results)
ANLS = sum(anls(r['pred'], r['gold']) for r in results)/len(results)
print('\n' + '='*50)
print(f'KET QUA (n={len(results)}): EM={EM*100:.1f}%  ANLS={ANLS*100:.1f}%')
print(f'KY VONG:            EM=38.3%   ANLS=55.9%')
ok = abs(EM*100-38.3) < 1.5 and abs(ANLS*100-55.9) < 1.5
print('✅ KHOP baseline' if ok else '⚠️ LECH - kiem lai (GPU/version/data)')
print('='*50)